In [ ]:
# Inference-only: no training ticket needed (CLAUDE.md's ticket rule
# applies to training runs; this only samples from an already-trained
# checkpoint). Costs Kaggle weekly quota, not the 10-hour training budget.
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

inputs = Path("/kaggle/input")
print("top-level /kaggle/input entries:", list(inputs.iterdir()))

roots = list((inputs / "nanowm-code").rglob("pyproject.toml"))
if not roots:
    roots = list(inputs.rglob("pyproject.toml"))
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one project root, found {len(roots)}: {roots}")
mounted = roots[0].parent
root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))

In [ ]:
subprocess.run([sys.executable, "scripts/preflight_gpu.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "lpips"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)

In [ ]:
assets_candidates = [p for p in Path("/kaggle/input").iterdir() if p.is_dir() and p.name != "nanowm-code"]
print("asset dataset candidates:", assets_candidates)
latest_hits = [p for c in assets_candidates for p in c.rglob("latest.json")]
manifest_hits = [p for c in assets_candidates for p in c.rglob("manifest.json")]
if not latest_hits or not manifest_hits:
    raise RuntimeError(f"latest.json hits={latest_hits}, manifest.json hits={manifest_hits} -- assets dataset not mounted as expected")
ckpt_dir = latest_hits[0].parent
data_dir = manifest_hits[0].parent
print("checkpoint dir:", ckpt_dir)
print("data dir:", data_dir)

subprocess.run([
    sys.executable, "scripts/run_m1_gate.py",
    "--checkpoint-dir", str(ckpt_dir),
    "--data-dir", str(data_dir),
    "--out", "/kaggle/working/m1_gate.json",
], check=True)

In [ ]:
import json
print(json.dumps(json.load(open("/kaggle/working/m1_gate.json")), indent=2))